# K-SPADA Model Training

[Project overview](../README.md) · [01 Data preparation](./01_data_preparation.ipynb) · [02 Model training](./02_model_training.ipynb) · [03 Prediction and visualization](./03_prediction_and_visualization.ipynb) · [04 Quantile binning](./04_binning_data_preparation.ipynb)

This tutorial trains K-SPADA on a generic prepared graph dataset. Dataset paths and model parameters are defined explicitly in the Notebook.

Run this notebook from the `tutorials/` directory after [data preparation](./01_data_preparation.ipynb). Prepared graphs are read from `../outputs/liver_cancer_data_preparation/`. Samples 4–15 are used for training and sample 16 for validation; model files are saved under `../outputs/model_training/example_model/`.

Executed on 2026-11-10 with **Python (KSPADA_env @ d05 GPU 3)**. The kernel ran on `d05` with GPU 3 selected.


In [1]:
from pathlib import Path
import anndata as ad
import torch
import KSPADA

PROJECT_ROOT = Path("..").resolve()
DATA_DIR = PROJECT_ROOT / "outputs" / "liver_cancer_data_preparation"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "model_training"
TRAIN_VALID_H5AD = DATA_DIR / "reference_data_train_4_5_6_7_8_9_10_11_12_13_14_15_valid_16_train_valid_CGraph_k_2.h5ad"
PROJECT_NAME = "example_model"

TRAIN_SAMPLES = ["4", "5", "6", "7", "8", "9", "10", "11", "12", "13", "14", "15"]
VALIDATION_SAMPLES = ["16"]
PATHWAY_NAME = "human_gobp"
LABEL_NAME = "Celltype"
MAX_GENES_PER_PATHWAY = 300
MAX_PATHWAY_TOKENS = 300
MASK_RATIO = 0.015
UNANNOTATED_TOKENS = 1
STANDARD_BATCH_SIZE = 8
EMBEDDING_DIM = 48
TRANSFORMER_DEPTH = 2
ATTENTION_HEADS = 4
LEARNING_RATE = 0.001
WEIGHT_DECAY = 5e-5
EPOCHS = 10
FINAL_LEARNING_RATE_RATIO = 0.01
GRAPH_BATCH_SIZE = 128
SPATIAL_NEIGHBORS = 5
LAMBDA_REGULARIZATION = 1e-6
RANDOM_SEED = 1

Matplotlib is building the font cache; this may take a moment.


## Step 1 — Select the GPU

**Execution result:** CUDA was available on an NVIDIA A100 80GB PCIe. With `CUDA_VISIBLE_DEVICES=3`, `cuda:0` refers to physical GPU 3.


In [2]:
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. Select a GPU-enabled KSPADA kernel before training.")

torch.cuda.set_device(0)
TRAINING_DEVICE = torch.device("cuda:0")

## Step 2 — Validate the prepared graph file

**Execution result:** graph fields and sample identifiers passed validation. The input contains **53,548 cells × 3,000 genes**: 49,406 training cells from samples 4–15 and 4,142 validation cells from sample 16.


In [3]:
if not TRAIN_VALID_H5AD.is_file():
    raise FileNotFoundError(TRAIN_VALID_H5AD)

KSPADA.validate_cgraph_fields(TRAIN_VALID_H5AD, mode="train")
training_adata = ad.read_h5ad(TRAIN_VALID_H5AD, backed="r")
actual_samples = set(training_adata.obs["batch"].astype(str).unique())
expected_samples = set(TRAIN_SAMPLES + VALIDATION_SAMPLES)
training_adata.file.close()
if actual_samples != expected_samples:
    raise ValueError(
        f"Sample mismatch: expected {sorted(expected_samples)}, found {sorted(actual_samples)}"
    )

/home/lixiangyu/anaconda3/envs/KSPADA_env/lib/python3.8/site-packages/anndata/_core/anndata.py:1838: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/lixiangyu/anaconda3/envs/KSPADA_env/lib/python3.8/site-packages/anndata/_core/anndata.py:1838: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


## Step 3 — Train the model

**Execution result:** all 10 epochs completed. Final training loss: **0.0291**; training accuracy: **99.08%**. Final validation loss: **0.9648**; validation accuracy: **84.92%**. The checkpoint was saved to `../outputs/model_training/example_model/model-9.pth`. Full training progress is retained in the cell output below.


In [4]:
model_directory = KSPADA.train_model(
    adata_input=TRAIN_VALID_H5AD,
    project=PROJECT_NAME,
    output_dir=OUTPUT_DIR,
    gmt_path=PATHWAY_NAME,
    label_name=LABEL_NAME,
    train_batches=TRAIN_SAMPLES,
    use_gcn=True,
    pre_weights="",
    max_g=MAX_GENES_PER_PATHWAY,
    max_gs=MAX_PATHWAY_TOKENS,
    mask_ratio=MASK_RATIO,
    n_unannotated=UNANNOTATED_TOKENS,
    batch_size=STANDARD_BATCH_SIZE,
    embed_dim=EMBEDDING_DIM,
    depth=TRANSFORMER_DEPTH,
    num_heads=ATTENTION_HEADS,
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    epochs=EPOCHS,
    lrf=FINAL_LEARNING_RATE_RATIO,
    batch_size_gcn=GRAPH_BATCH_SIZE,
    neighbor=SPATIAL_NEIGHBORS,
    lambda_reg=LAMBDA_REGULARIZATION,
    random_seed=RANDOM_SEED,
)

/home/lixiangyu/anaconda3/envs/KSPADA_env/lib/python3.8/site-packages/anndata/_core/anndata.py:1838: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


batch_size_gcn: 128
lambda_reg: 1e-06
neighbor: 5
num_head: 4
depth: 2
embed_dim: 48
weight_decay: 5e-05
random_seed: 1
cuda:0


torch.Size([2, 247030])
torch.Size([2, 20710])
enhance:  torch.Size([2, 247030])
enhance:  torch.Size([2, 20710])


edge_index_train shape before merge: torch.Size([2, 551343])
edge_index_valid shape before merge: torch.Size([2, 38994])
edge_index_train shape after merge: torch.Size([2, 553808])
edge_index_valid shape after merge: torch.Size([2, 39201])


Mask loaded!
use_gcn:True
use_position_encode: False
embed_dim_heads: 48
num_heads: 4


Model builded!


... (4,834 outputs left out)

[valid epoch 9] loss: 0.976, acc: 0.848:  92%|█████████▏| 57/62 [00:01<00:00, 34.97it/s]

[valid epoch 9] loss: 0.966, acc: 0.849:  98%|█████████▊| 61/62 [00:01<00:00, 35.29it/s]

[valid epoch 9] loss: 0.965, acc: 0.849: 100%|██████████| 62/62 [00:01<00:00, 31.97it/s]

Training finished!


## Notes

- Replace the sample names with values present in the prepared graph file.
- Training and prediction must use the same embedding dimension, Transformer depth, attention-head count, and graph settings.